In [1]:
import json
import pandas as pd
import numpy as np
import psycopg2
import pyodbc
import mariadb

In [2]:
username = "scoring"
password = "idkltb93e0eomejp"
host = 'spectral-msql-jul-24-backup-do-user-2276924-0.b.db.ondigitalocean.com'
port = 25060
database="farmlabv3_live"
# schema="farmlabv3_live"
# schema="historical"
def get_db_cursor():
    try:
        conn = mariadb.connect(
            user=username,
            password=password,
            host=host,
            port=port,
            database=database
    
        )
        return conn
    except mariadb.Error as e:
        print(f"Error connecting to MariaDB Platform: {e}")
        sys.exit(1)

conn = get_db_cursor()
cur = conn.cursor()

In [3]:
conn_lims = pyodbc.connect("Driver={SQL Server};"
                            "Server=192.168.5.18\CROPNUT;"
                            "Database=cropnuts;"
                            "uid=thomasTsuma;pwd=GR^KX$uRe9#JwLc6")
cursor_lims = conn_lims.cursor()

In [4]:
with open("outputFiles/renaming.json") as file:
    renaming_file = json.load(file)

In [5]:
renaming_file

{'TEST-DS1-1800': 'DIA-HO-0233',
 'TEST-DS1-1801': 'OCP-T1-25059',
 'TEST-DS1-1802': 'RWA-TS-1402'}

In [6]:
df = pd.read_excel("inputFiles/batch-approval-batch_9507 (14).xlsx",sheet_name="Scoring Results",header=1)

In [7]:
df[['Sample Code','Crop','pH', 'Exchangeable K','Available P','Organic Matter']]

,Sample Code,Crop,pH,Exchangeable K,Available P,Organic Matter
0,TEST-DS1-1660,Maize,8.0,460,>50,1.6
1,TEST-DS1-1661,Maize,6.7,100,>50,2.0
2,TEST-DS1-1663,Maize,7.2,2000,30-120,5.9
3,TEST-DS1-1664,Maize,6.9,130,30-120,2.6
4,TEST-DS1-1665,Maize,5.4,60,30-120,3.5
...,...,...,...,...,...,...
70,TEST-DS1-1767,Tomatoes (Open field),5.6,140,>50,2.9
71,TEST-DS1-1768,Tomatoes (Open field),7.5,960,30-50,3.9
72,TEST-DS1-1769,Tomatoes (Open field),7.4,320,30-50,4.7
73,TEST-DS1-1770,Tomatoes (Open field),5.7,140,30-50,5.0


In [8]:
renaming_file.values()

dict_values(['OCP-OG-00331', 'OCP-GH-BR1071', 'AGR-FL-10113', 'OCP-KR-011', 'OCP-OG-00290', 'OCP-GH-BR2390', 'OCP-OG-00346', 'OCP-OG-00444', 'PAG-FL-2135', 'PAG-FL-2106', 'OCP-T1-21554', 'RWA-TS-0199', 'AGR-FL-10030', 'OCP-SN-9878', 'OCP-SN-9878', 'OCP-OG-00346', 'OCP-NS-00156', 'DIA-HO-0348', 'AGR-FL-10066', 'OCP-OG-00317', 'OCP-GH-BR2390', 'OCP-OG-00346', 'OCP-NS-00156', 'DIA-HO-0348', 'AGR-FL-10066', 'RWA-TS-0741', 'OCP-GH-BR2390', 'OCP-OG-00331', 'OCP-NS-00125', 'OCP-GH-BR1964', 'AGB-FL-0458', 'OCP-OG-00290', 'DIA-BU-0017', 'OCP-SN-9888', 'OCP-SN-9878', 'OCP-OG-00331', 'OCP-GH-BR1071', 'AGR-FL-10113', 'AGB-FL-0458', 'OCP-OG-00290', 'OCP-GH-BR2390', 'OCP-NS-00125', 'OCP-T1-25816', 'AGR-FL-10113', 'OCP-GH-BR3456', 'OCP-NS-00088', 'OCP-GH-BR3614', 'OCP-KR-002', 'OCP-KB-00202', 'RWA-TS-0074', 'AGB-FL-0458', 'OCP-OG-00346', 'OCP-NS-00156', 'OCP-GH-BR1964', 'OCP-GH-BR3493', 'OCP-OG-00317', 'OCP-GH-BR2390', 'OCP-OG-00346', 'OCP-NS-00128', 'PAG-FL-2135', 'OCP-GH-BR3413', 'RWA-TS-0272', 'OC

In [9]:
fl_preds = pd.read_sql(f"SELECT barcode, chemical_name, score, class_score FROM ScoringResult sr INNER JOIN MeasuredChemical mc ON mc.chemical_id = sr.chemical_id INNER JOIN SpectralSample ss ON ss.spectral_sample_id = sr.spectral_sample_id WHERE ss.barcode IN {tuple(renaming_file.values())}",con=conn) 

C:\Users\tsuma.thomas\AppData\Local\Temp\ipykernel_18340\1862013524.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fl_preds = pd.read_sql(f"SELECT barcode, chemical_name, score, class_score FROM ScoringResult sr INNER JOIN MeasuredChemical mc ON mc.chemical_id = sr.chemical_id INNER JOIN SpectralSample ss ON ss.spectral_sample_id = sr.spectral_sample_id WHERE ss.barcode IN {tuple(renaming_file.values())}",con=conn)


In [10]:
fl_preds

,barcode,chemical_name,score,class_score
0,AGB-FL-0458,phosphorus,58.777802,optimum
1,AGB-FL-0458,ph,6.839808,None
2,AGB-FL-0458,exchangeable_acidity,0.126667,None
3,AGB-FL-0458,calcium,1094.477417,None
4,AGB-FL-0458,magnesium,168.112076,None
...,...,...,...,...
835,RWA-TS-0741,cec,51.409775,None
836,RWA-TS-0741,sand,58.941525,None
837,RWA-TS-0741,silt,29.777306,None
838,RWA-TS-0741,clay,17.888165,None


In [11]:
fl_preds_pivot = fl_preds.pivot_table(index='barcode', columns='chemical_name', values='score')

In [12]:
fl_preds_pivot = fl_preds_pivot[['ph','potassium','phosphorus','organic_carbon']]

In [13]:
fl_preds_pivot = fl_preds_pivot.reindex(renaming_file.values())

In [14]:
df = df[['Sample Code','Crop','pH', 'Exchangeable K','Available P','Organic Matter']]

In [15]:
fl_preds_pivot = fl_preds_pivot.reset_index()

In [16]:
df['barcode'] = df['Sample Code'].apply(lambda x : renaming_file[x] if x in [j for j in renaming_file.keys()] else None)

In [17]:
df

,Sample Code,Crop,pH,Exchangeable K,Available P,Organic Matter,barcode
0,TEST-DS1-1660,Maize,8.0,460,>50,1.6,OCP-OG-00331
1,TEST-DS1-1661,Maize,6.7,100,>50,2.0,OCP-GH-BR1071
2,TEST-DS1-1663,Maize,7.2,2000,30-120,5.9,AGR-FL-10113
3,TEST-DS1-1664,Maize,6.9,130,30-120,2.6,OCP-KR-011
4,TEST-DS1-1665,Maize,5.4,60,30-120,3.5,OCP-OG-00290
...,...,...,...,...,...,...,...
70,TEST-DS1-1767,Tomatoes (Open field),5.6,140,>50,2.9,OCP-T2-21773
71,TEST-DS1-1768,Tomatoes (Open field),7.5,960,30-50,3.9,PAG-FL-2135
72,TEST-DS1-1769,Tomatoes (Open field),7.4,320,30-50,4.7,OCP-GH-BR1806
73,TEST-DS1-1770,Tomatoes (Open field),5.7,140,30-50,5.0,OCP-T1-23586


In [18]:
fl_preds_pivot['organic_matter'] = fl_preds_pivot['organic_carbon'] * 1.74

In [19]:
fl_preds_pivot

chemical_name,barcode,ph,potassium,phosphorus,organic_carbon,organic_matter
0,OCP-OG-00331,8.012294,459.881989,112.632095,0.943631,1.641918
1,OCP-GH-BR1071,6.695257,99.663826,105.358719,1.138765,1.981451
2,AGR-FL-10113,7.167424,1997.551758,50.931343,3.449111,6.001454
3,OCP-KR-011,6.896716,122.253014,67.931068,1.505970,2.620388
4,OCP-OG-00290,5.385703,58.958057,73.479050,2.028629,3.529815
...,...,...,...,...,...,...
73,OCP-T2-21773,5.638970,132.994171,111.148125,1.690977,2.942300
74,PAG-FL-2135,7.499644,959.006287,53.893955,2.285057,3.976000
75,OCP-GH-BR1806,7.446612,319.096100,52.756367,2.725119,4.741707
76,OCP-T1-23586,5.691709,130.908844,55.151703,2.886382,5.022304


In [20]:
df_merged = pd.merge(df, fl_preds_pivot, on='barcode', how='inner')

In [21]:
df_merged = df_merged.drop_duplicates(subset='barcode')

In [22]:
df_merged

,Sample Code,Crop,pH,Exchangeable K,Available P,Organic Matter,barcode,ph,potassium,phosphorus,organic_carbon,organic_matter
0,TEST-DS1-1660,Maize,8.0,460,>50,1.6,OCP-OG-00331,8.012294,459.881989,112.632095,0.943631,1.641918
4,TEST-DS1-1661,Maize,6.7,100,>50,2.0,OCP-GH-BR1071,6.695257,99.663826,105.358719,1.138765,1.981451
6,TEST-DS1-1663,Maize,7.2,2000,30-120,5.9,AGR-FL-10113,7.167424,1997.551758,50.931343,3.449111,6.001454
9,TEST-DS1-1664,Maize,6.9,130,30-120,2.6,OCP-KR-011,6.896716,122.253014,67.931068,1.505970,2.620388
10,TEST-DS1-1665,Maize,5.4,60,30-120,3.5,OCP-OG-00290,5.385703,58.958057,73.479050,2.028629,3.529815
14,TEST-DS1-1667,Maize,7.0,400,10-30,8.1,OCP-GH-BR2390,6.998568,396.048126,37.738758,4.694140,8.167804
21,TEST-DS1-1669,Beans,7.8,550,>50,2.1,OCP-OG-00346,7.825081,548.403381,100.320183,1.243635,2.163926
27,TEST-DS1-1670,Beans,6.6,140,>50,3.0,OCP-OG-00444,6.630710,130.753006,116.061264,1.722611,2.997343
29,TEST-DS1-1672,Beans,7.5,960,30-50,3.9,PAG-FL-2135,7.499644,959.006287,53.893955,2.285057,3.976000
33,TEST-DS1-1673,Beans,6.9,380,30-50,3.3,PAG-FL-2106,6.878088,376.418915,58.480995,1.892390,3.292758


In [23]:
df_merged.to_csv("outputFiles/verification_predictions.csv")

In [24]:
df_preds = pd.read_csv("outputFiles/verification_predictions.csv")

In [25]:
df_colors = pd.read_csv("outputFiles/verification_class_color_codes.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'outputFiles/verification_class_color_codes.csv'

In [ ]:
df_colors

In [ ]:
mismatched = df_colors.loc[df_colors.potassium == False]

In [ ]:
len(mismatched)

In [ ]:
mismatched.barcode

In [ ]:
actual_codes = [ renaming_file[i] for i in mismatched.barcode.values ]

In [ ]:
actual_codes

In [ ]:
scores_fl = pd.read_sql(f"SELECT * FROM ScoringResult sr INNER JOIN SpectralSample ss ON ss.spectral_sample_id = sr.spectral_sample_id WHERE ss.barcode IN {tuple(actual_codes)}", con=conn)

In [ ]:
scores_fl

In [ ]:
scores_fl.columns

In [ ]:
scores_fl.batch_date.min()

In [ ]:
scores_fl.batch_date.max()